In [1]:
# Week 12 Homework: Text to Image, 5 models
# Task: Text to Image (prompt in, picture out).
# Why: I like to see how one short sentence becomes a picture, and new models come out very fast.
# I run 5 different models on the same 4 prompts, the same seed and the same number of steps.
# Metric: CLIP score (higher is better) and seconds per image.
# Bonus: SDXL-Turbo (fast, distilled) against SDXL (full).
# Run this on Colab with a T4 GPU. All 5 models run inside Colab.

In [2]:
# Setup: install the libraries. Colab already has torch.
!pip install -q -U diffusers transformers accelerate sentencepiece protobuf ftfy beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.


In [3]:
# Setup: imports, seed and the CLIP judge.
import warnings
warnings.filterwarnings("ignore")

import gc
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from diffusers import AutoPipelineForText2Image
from transformers import CLIPModel, CLIPProcessor
from huggingface_hub import scan_cache_dir

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Go to Runtime > Change runtime type > T4 GPU.")

# CLIP gives one score for each (prompt, image) pair.
CLIP_ID = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(CLIP_ID).to(device).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)

[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.


Device: cuda
GPU: Tesla T4


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [4]:
# Setup: prompts and models.
# Every model gets the same prompts, the same seed and the same steps.
# Prompt 0 is from class. Prompt 1 has text in the picture and prompt 2 has hands. Models often fail on these.
prompts = [
    "a small robot watering a bonsai tree, studio lighting, highly detailed",
    "a red neon sign that says OPEN 24 HOURS on a rainy street at night",
    "a close up photo of two hands holding a coffee cup",
    "a cat wearing a space suit riding a bicycle on the moon",
]
STEPS = 25

# The 5 models. Each one has a different design.
# size = normal image size of the model, cfg = guidance scale that the model authors suggest.
models = [
    {"name": "SD 1.5", "id": "stable-diffusion-v1-5/stable-diffusion-v1-5",
     "arch": "U-Net + CLIP text encoder", "params": "0.9B", "size": 512, "cfg": 7.5,
     # The safety checker returns a black image for some prompts. Turn it off so every model is judged fairly.
     "kwargs": {"safety_checker": None, "requires_safety_checker": False}},
    {"name": "SDXL", "id": "stabilityai/stable-diffusion-xl-base-1.0",
     "arch": "bigger U-Net + 2 CLIP text encoders", "params": "3.5B", "size": 1024, "cfg": 7.5, "kwargs": {"variant": "fp16"}},
    {"name": "PixArt-Sigma", "id": "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS",
     "arch": "Diffusion Transformer + T5 text encoder", "params": "0.6B + 4.7B T5", "size": 1024, "cfg": 4.5, "kwargs": {}},
    {"name": "Kandinsky 2.2", "id": "kandinsky-community/kandinsky-2-2-decoder",
     "arch": "prior + U-Net (image embedding)", "params": "about 1.2B U-Net + prior", "size": 768, "cfg": 4.0, "kwargs": {}},
    {"name": "Sana", "id": "Efficient-Large-Model/Sana_600M_512px_diffusers",
     "arch": "Linear Diffusion Transformer + Gemma 2 text encoder", "params": "0.6B + 2B Gemma", "size": 512, "cfg": 4.5, "kwargs": {}, "dtype": torch.bfloat16},
]

info = pd.DataFrame(models)[["name", "id", "arch", "params"]]
info["open_source"] = "yes (open weights)"
info["ran_on"] = "Colab T4 GPU"
info

,name,id,arch,params,open_source,ran_on
0,SD 1.5,stable-diffusion-v1-5/stable-diffusion-v1-5,U-Net + CLIP text encoder,0.9B,yes (open weights),Colab T4 GPU
1,SDXL,stabilityai/stable-diffusion-xl-base-1.0,bigger U-Net + 2 CLIP text encoders,3.5B,yes (open weights),Colab T4 GPU
2,PixArt-Sigma,PixArt-alpha/PixArt-Sigma-XL-2-512-MS,Diffusion Transformer + T5 text encoder,0.6B + 4.7B T5,yes (open weights),Colab T4 GPU
3,Kandinsky 2.2,kandinsky-community/kandinsky-2-2-decoder,prior + U-Net (image embedding),about 1.2B U-Net + prior,yes (open weights),Colab T4 GPU
4,Sana,Efficient-Large-Model/Sana_600M_512px_diffusers,Linear Diffusion Transformer + Gemma 2 text en...,0.6B + 2B Gemma,yes (open weights),Colab T4 GPU


In [ ]:
# Run: load one model at a time, make 4 images, score them, then free the GPU.
# If a model fails, we print the error and go on to the next one.
records = []
failed = []
for m in models:
    print("Loading", m["name"], "...")
    pipe = None
    try:
        pipe = AutoPipelineForText2Image.from_pretrained(m["id"], torch_dtype=m.get("dtype", torch.float16), **m["kwargs"])
        if m["name"] == "Sana":
            # Sana wants bfloat16 for the text encoder and the vae, but the T4 GPU has no bfloat16 support
            # ("unable to find an engine"). Use float32 for those two parts and float16 for the transformer.
            pipe.text_encoder.to(torch.float32)
            pipe.vae.to(torch.float32)
            pipe.transformer.to(torch.float16)
        pipe = pipe.to(device)
        pipe.set_progress_bar_config(disable=True)

        # Colab disk is small. The model is in memory now, so delete its downloaded files.
        cache = scan_cache_dir()
        old = [rev.commit_hash for repo in cache.repos
               if repo.repo_id.split("/")[0] == m["id"].split("/")[0] for rev in repo.revisions]
        cache.delete_revisions(*old).execute()

        # Warm-up run, so the first timing is not slow.
        pipe(prompts[0], height=m["size"], width=m["size"], num_inference_steps=1, guidance_scale=m["cfg"])

        for i, p in enumerate(prompts):
            generator = torch.Generator(device).manual_seed(SEED)
            t0 = time.time()
            img = pipe(p, height=m["size"], width=m["size"], num_inference_steps=STEPS,
                       guidance_scale=m["cfg"], generator=generator).images[0]
            seconds = time.time() - t0

            inputs = clip_processor(text=[p], images=[img], return_tensors="pt", padding=True).to(device)
            with torch.no_grad():
                out = clip_model(**inputs)
            clip = 100 * max(torch.cosine_similarity(out.image_embeds, out.text_embeds).item(), 0)

            records.append({"model": m["name"], "prompt_id": i, "seconds": seconds, "clip": clip, "image": img})
            print(f"  prompt {i}: {seconds:.1f}s, CLIP {clip:.1f}")
    except Exception as e:
        print("FAILED:", m["name"], "->", e)
        failed.append(m["name"])

    del pipe
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Run: all outputs side by side. Rows are prompts, columns are models.
names = list(dict.fromkeys(r["model"] for r in records))
fig, axes = plt.subplots(len(prompts), len(names), figsize=(3.3 * len(names), 3.6 * len(prompts)), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for r in records:
    ax = axes[r["prompt_id"], names.index(r["model"])]
    ax.imshow(r["image"])
    ax.set_title(f'{r["model"]}\nCLIP {r["clip"]:.1f}, {r["seconds"]:.1f}s', fontsize=9)
plt.tight_layout()
plt.show()

for i, p in enumerate(prompts):
    print(f"Prompt {i}: {p}")

In [ ]:
# Score: CLIP score and time for each model.
# CLIP checks if the picture matches the prompt. It does not check spelling or hands.
df = pd.DataFrame(records).drop(columns="image")

print("CLIP score for each prompt:")
display(df.pivot(index="model", columns="prompt_id", values="clip").loc[names].round(1))

summary = df.groupby("model", sort=False).agg(
    clip_mean=("clip", "mean"), clip_std=("clip", "std"), seconds_mean=("seconds", "mean")).round(2)
summary

In [ ]:
# Score: charts.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(summary.index, summary.clip_mean, yerr=summary.clip_std.fillna(0), color="#4C72B0")
axes[0].set_title("Mean CLIP score (higher is better)")
axes[1].bar(summary.index, summary.seconds_mean, color="#DD8452")
axes[1].set_title("Seconds per image (lower is better)")
for ax in axes:
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
# Compare: best, fastest, most stable, and the worst result.
print("Best CLIP score :", summary.clip_mean.idxmax(), summary.clip_mean.max())
print("Fastest         :", summary.seconds_mean.idxmin(), summary.seconds_mean.min(), "s")
print("Most stable     :", summary.clip_std.idxmin(), "(lowest CLIP std between prompts)")
print("Failed to run   :", failed if failed else "none")

# Failure case: the image with the lowest CLIP score.
worst = df.loc[df["clip"].idxmin()]
worst_img = [r["image"] for r in records if r["model"] == worst["model"] and r["prompt_id"] == worst["prompt_id"]][0]
plt.figure(figsize=(4, 4))
plt.imshow(worst_img)
plt.axis("off")
plt.title(f'Worst: {worst["model"]}, prompt {int(worst["prompt_id"])}, CLIP {worst["clip"]:.1f}', fontsize=9)
plt.show()
print("Prompt:", prompts[int(worst["prompt_id"])])

# CLIP cannot see spelling mistakes. Look at prompt 1 (the sign) and prompt 2 (the hands) in the grid above.

In [ ]:
# Bonus: SDXL-Turbo is SDXL made faster (distilled). It needs only 1 to 4 steps and no guidance.
# We compare it with the full SDXL (25 steps). SDXL-Turbo works best at 512 pixels, SDXL at 1024.
pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16").to(device)
pipe.set_progress_bar_config(disable=True)
pipe(prompts[0], height=512, width=512, num_inference_steps=1, guidance_scale=0.0)  # warm-up

turbo_records = []
for steps in [1, 4]:
    for i, p in enumerate(prompts):
        generator = torch.Generator(device).manual_seed(SEED)
        t0 = time.time()
        img = pipe(p, height=512, width=512, num_inference_steps=steps, guidance_scale=0.0, generator=generator).images[0]
        seconds = time.time() - t0

        inputs = clip_processor(text=[p], images=[img], return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            out = clip_model(**inputs)
        clip = 100 * max(torch.cosine_similarity(out.image_embeds, out.text_embeds).item(), 0)

        turbo_records.append({"model": f"SDXL-Turbo {steps} step", "prompt_id": i, "seconds": seconds, "clip": clip, "image": img})

del pipe
gc.collect()
torch.cuda.empty_cache()

# Table: SDXL (25 steps) against SDXL-Turbo.
bonus_records = [r for r in records if r["model"] == "SDXL"] + turbo_records
bonus = pd.DataFrame(bonus_records).drop(columns="image").groupby("model", sort=False).agg(
    clip_mean=("clip", "mean"), seconds_mean=("seconds", "mean")).round(2)
bonus["images_per_second"] = (1 / bonus.seconds_mean).round(2)
display(bonus)

# Pictures: rows are prompts, columns are models.
bnames = list(dict.fromkeys(r["model"] for r in bonus_records))
fig, axes = plt.subplots(len(prompts), len(bnames), figsize=(3.3 * len(bnames), 3.6 * len(prompts)), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for r in bonus_records:
    ax = axes[r["prompt_id"], bnames.index(r["model"])]
    ax.imshow(r["image"])
    ax.set_title(f'{r["model"]}\nCLIP {r["clip"]:.1f}, {r["seconds"]:.1f}s', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Summary (this text is made from the results above).
best = summary.clip_mean.idxmax()
fast = summary.seconds_mean.idxmin()
speedup = bonus.loc["SDXL", "seconds_mean"] / bonus.loc["SDXL-Turbo 1 step", "seconds_mean"]
clip_drop = bonus.loc["SDXL", "clip_mean"] - bonus.loc["SDXL-Turbo 1 step", "clip_mean"]

print(f"Task: Text to Image. {len(names)} models, {len(prompts)} prompts, seed {SEED}, {STEPS} steps.")
print("Models:", ", ".join(names))
print("Metric: CLIP score (higher is better) and seconds per image, plus my own look at the pictures.")
print(f"Best CLIP score: {best} ({summary.clip_mean[best]:.1f}). Fastest: {fast} ({summary.seconds_mean[fast]:.1f}s per image).")
print(f"Worst single result: {worst['model']}, prompt {int(worst['prompt_id'])}, CLIP {worst['clip']:.1f}.")
print(f"Bonus: SDXL-Turbo with 1 step is {speedup:.0f} times faster than SDXL with 25 steps, and CLIP changes by {-clip_drop:+.1f}.")

In [ ]:
# My notes and conclusion (write these after you look at the pictures).
# Best picture quality:
# Text in the sign (prompt 1), which model spelled it right:
# Hands (prompt 2), which model was best:
# Failure case, and why I think it failed:
# What surprised me:
# The model I would choose for a real job, and why: